# Fine-tuning QLoRA de Qwen2.5-7B para clasificar funciones de cita

Motivo: con prompting (few-shot) el modelo obtuvo macro-F1 ≈ 0.35 contra las anotaciones humanas; confunde sistemáticamente las convenciones de anotación (p. ej. sobre-predice *Identification of the Originator*). Aquí se **ajusta el modelo con las 2 700 citas anotadas**.

Diseño:
- **Salida de una sola letra** (A–K = 9 funciones + `Ambiguo` + `No evaluable`). No se genera texto: se leen los logits de las 11 letras en un único *forward pass*. Eso da la **distribución de probabilidad** (confianza real, calibrable) y es mucho más rápido que generar JSON.
- Se reservan ~15 % de las anotaciones humanas como **validación** (no se usan para entrenar).
- Las clases raras se sobremuestrean hasta un mínimo (`OVERSAMPLE_MIN`).
- Opcional (`FINAL_FIT_ALL`): una vez fijados los hiperparámetros, reentrenar con las 2 700 completas.
- Después se clasifican las 18 000 citas con *checkpoint* y se exporta un CSV.

> Las etiquetas generadas siguen siendo **pre-anotación**, no gold humano. Revisa a mano las de confianza Baja. Las 2 700 anotaciones de entrenamiento las hizo una sola persona.

## 0. Instalación (una sola vez, fuera del notebook)

```bash
cd microproyecto-local-citation/notebooks/qlora-training
uv venv .venv-qlora --python 3.12
source .venv-qlora/bin/activate

# PyTorch con CUDA (ajusta cu124 a la versión que soporte tu driver; `nvidia-smi` muestra la máxima)
uv pip install torch --index-url https://download.pytorch.org/whl/cu124
uv pip install transformers peft accelerate bitsandbytes pandas ipykernel

# Registrar el kernel y elegirlo en Jupyter/VS Code ("Python (qlora)")
python -m ipykernel install --user --name qlora --display-name "Python (qlora)"

# Comprobación
python -c "import torch, bitsandbytes; print(torch.cuda.is_available(), torch.cuda.get_device_name(0))"
```

**Libera la GPU antes de entrenar.** Ollama mantiene el modelo cargado hasta 30 min (~5-6 GB de los 12 GB):

```bash
ollama stop qwen2.5:7b-instruct      # descarga el modelo de la VRAM
# o, para parar todo el servicio:  sudo systemctl stop ollama
nvidia-smi                           # comprueba que quedan >10 GB libres
```

El modelo base (`Qwen/Qwen2.5-7B-Instruct`, ~15 GB en fp16) se descarga de Hugging Face la primera vez y se cuantiza a 4 bits al cargarlo (~5.5 GB de VRAM).

## 1. Configuración y datos

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # menos fragmentación de VRAM; debe fijarse antes de importar torch
import json, random, time
from collections import Counter
from pathlib import Path

import pandas as pd

BASE = Path('.')
GOLD_CSV   = BASE / 'ANOTACION_UNICA_2700_CITAS_REALES_ANOTADO.csv'
TARGET_CSV = BASE / 'ANOTACION_18000_CITAS_REALES.csv'

# False: entrenar con ~85 % y validar con el resto (elegir hiperparámetros).
# True: reentrenar con las 2 700 completas (modelo final); entonces la validación interna deja de ser válida.
FINAL_FIT_ALL = False

SUF = '_v1_all' if FINAL_FIT_ALL else '_v1'
OUT_CSV     = BASE / f'ANOTACION_18000_CITAS_REALES_QLORA{SUF}.csv'
ADAPTER_DIR = BASE / f'qlora_citas_adapter{SUF}'
TRAIN_DIR   = BASE / f'qlora_citas_trainer{SUF}'
SPLIT_JSON  = BASE / 'qlora_split_v1.json'
VAL_JSONL   = BASE / 'qlora_val_preds_v1.jsonl'
RUN_JSONL   = BASE / f'qlora_18000_preds{SUF}.jsonl'

BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
SEED = 42

# datos
VAL_FRAC = 0.15            # fracción de las anotaciones humanas reservada para validar
OVERSAMPLE_MIN = 100       # las clases con menos ejemplos de entrenamiento se repiten hasta este mínimo
MAX_LEN = 1024             # tokens máximos por ejemplo (se descartan los más largos)
ABSTRACT_CHARS = 400

# entrenamiento (BATCH=1 con GRAD_ACCUM=16 = lote efectivo de 16 y usa menos VRAM)
SAVE_STEPS = 25            # checkpoint cada N pasos: si se interrumpe, se pierden minutos, no horas
EPOCHS = 2
BATCH = 1
GRAD_ACCUM = 16
LR = 2e-4
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05

# inferencia
INFER_BATCH = 8
CONF_HIGH, CONF_MID = 0.80, 0.50    # umbrales de probabilidad para Alta / Media (calíbralos con la validación)

In [ ]:
FUNCTIONS = {
    'Background': 'Antecedentes generales del tema: contexto, trasfondo o historia de un campo; revisa literatura existente.',
    'Gap': 'Identifica un vacío o limitación en el trabajo previo y justifica la elección del tema.',
    'Basis': 'Fundamenta una afirmación o premisa; punto de partida sobre el que el trabajo actual construye o expande.',
    'Comparison': 'Compara explícitamente enfoques: el trabajo actual con los citados, o los citados entre sí.',
    'Application': 'Utiliza directamente, SIN modificar, un método, técnica, herramienta o dato del trabajo citado.',
    'Improvement / Modification': 'Adapta, mejora o extiende el método o herramienta citados.',
    'Evidence': 'Sustenta con evidencia o resultados: respalda afirmaciones, hipótesis o hallazgos.',
    'Identification of the Originator': 'Atribuye el origen de una idea, concepto, método o teoría a la obra pionera.',
    'Further Reading': 'Invita a leer más detalles: remite a fuentes externas para mayor profundidad.',
}
LABELS = list(FUNCTIONS) + ['Ambiguo', 'No evaluable']
LETTERS = 'ABCDEFGHIJK'[:len(LABELS)]
L2LET = dict(zip(LABELS, LETTERS)); LET2L = dict(zip(LETTERS, LABELS))
DESCR = dict(FUNCTIONS, **{'Ambiguo': 'No se puede inferir la función: hay dos o más lecturas posibles o el contexto está truncado.',
                           'No evaluable': 'El contexto no aporta ninguna información sobre la función de la cita.'})

COLS = {'ctx': 'Contexto TARGETCIT', 'title': 'Título citado', 'abs': 'Resumen citado',
        'ev': '¿Evaluable?', 'fn': 'Función principal', 'conf': 'Confianza', 'just': 'Justificación / duda'}

SYSTEM_PROMPT = ('Clasifica la función de la cita marcada TARGETCIT (OTHERCIT son otras citas) en un artículo de lingüística computacional. '
                 'Responde solo con la letra.\n'
                 + '\n'.join(f'{L2LET[k]}) {k}' for k in LABELS))

def user_message(row):
    title = row[COLS['title']].strip() or '(sin título)'
    return f"CONTEXTO:\n{row[COLS['ctx']].strip()}\n\nTRABAJO CITADO: {title}\nRESUMEN: {row[COLS['abs']].strip()[:ABSTRACT_CHARS]}"

def gold_label(row):
    return row[COLS['fn']] if row[COLS['ev']] == 'Evaluable' else row[COLS['ev']]

gold = pd.read_csv(GOLD_CSV, encoding='utf-8-sig', dtype=str, keep_default_na=False)
target = pd.read_csv(TARGET_CSV, encoding='utf-8-sig', dtype=str, keep_default_na=False)
gold['label'] = gold.apply(gold_label, axis=1)
assert set(gold['label']) <= set(LABELS), set(gold['label']) - set(LABELS)
print(len(gold), len(target)); print(gold['label'].value_counts().to_string())

### Partición entrenamiento / validación (por artículo citante, estratificada, guardada en disco)

Los contextos se agrupan por **artículo citante** (el prefijo de `context_id`), de modo que ningún citante queda a la vez en entrenamiento y validación. Entre muchas particiones aleatorias de grupos se elige la que deja la proporción de cada etiqueta más cercana al 15 %. La partición se guarda en `qlora_split_v1.json` para que sea **idéntica** entre ejecuciones.

También se comprueba el aislamiento frente al **Test Gold**: ningún contexto de entrenamiento puede compartir citante con él. Los artículos citados repetidos se permiten y se reportan aparte.

In [ ]:
import numpy as np

gold['citing'] = gold['context_id'].map(lambda c: c.rsplit('_', 2)[0])
gold['cited']  = gold['context_id'].map(lambda c: c.rsplit('_', 2)[1])

def make_split(n_trials=3000):
    keys = sorted(gold['citing'].unique())
    classes = sorted(gold['label'].unique())
    idx = {k: i for i, k in enumerate(keys)}
    M = np.zeros((len(keys), len(classes)))
    for cit, lab in zip(gold['citing'], gold['label']):
        M[idx[cit], classes.index(lab)] += 1
    sizes = M.sum(1); tot = M.sum(0); target = VAL_FRAC * len(gold)
    best, best_score = None, np.inf
    for t in range(n_trials):
        order = np.random.default_rng(SEED + t).permutation(len(keys))
        k = int(np.searchsorted(np.cumsum(sizes[order]), target)) + 1
        val_n = M[order[:k]].sum(0)
        score = (np.abs(val_n - VAL_FRAC * tot) / tot).sum()
        if score < best_score: best, best_score = order[:k], score
    val_cit = {keys[i] for i in best}
    is_val = gold['citing'].isin(val_cit)
    return {'train': sorted(gold.loc[~is_val, 'context_id']), 'val': sorted(gold.loc[is_val, 'context_id']),
            'method': f'agrupado por citante, mejor de {n_trials} particiones, VAL_FRAC={VAL_FRAC}, SEED={SEED}'}

if SPLIT_JSON.exists():
    split = json.loads(SPLIT_JSON.read_text())
else:
    split = make_split(); SPLIT_JSON.write_text(json.dumps(split))
train_df = gold[gold['context_id'].isin(set(split['train']))]
val_df   = gold[gold['context_id'].isin(set(split['val']))]

# --- comprobaciones de aislamiento
assert not set(split['train']) & set(split['val'])
assert set(split['train']) | set(split['val']) == set(gold['context_id'])
assert not set(train_df['citing']) & set(val_df['citing']), 'un citante aparece en train y val'
val_df = val_df.assign(cited_visto=val_df['cited'].isin(set(train_df['cited'])))
print('train', len(train_df), '| val', len(val_df), '|', split.get('method', ''))
print(f"val con citado visto en train: {val_df['cited_visto'].sum()} | no visto: {(~val_df['cited_visto']).sum()}")
print(pd.DataFrame({'train': train_df['label'].value_counts(), 'val': val_df['label'].value_counts()}).fillna(0).astype(int).to_string())


def oversample(df):
    rng = random.Random(SEED); parts = [df]
    for lab, g in df.groupby('label'):
        if len(g) < OVERSAMPLE_MIN:
            parts.append(g.sample(OVERSAMPLE_MIN - len(g), replace=True, random_state=rng.randint(0, 10**6)))
    return pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)

fit_df = gold if FINAL_FIT_ALL else train_df
train_os = oversample(fit_df)
print('entrena con', 'las 2 700 completas' if FINAL_FIT_ALL else 'train', '| tras sobremuestreo:', len(train_os)); print(train_os['label'].value_counts().to_string())

## 2. Métricas (se usan para validación)

In [ ]:
def report(y_true, y_pred, probs=None):
    labs = [l for l in LABELS if l in set(y_true)]
    acc = sum(a == b for a, b in zip(y_true, y_pred)) / len(y_true)
    rows = []
    for c in labs:
        tp = sum(a == b == c for a, b in zip(y_true, y_pred))
        fp = sum(b == c and a != c for a, b in zip(y_true, y_pred))
        fn = sum(a == c and b != c for a, b in zip(y_true, y_pred))
        p = tp / (tp + fp) if tp + fp else 0; r = tp / (tp + fn) if tp + fn else 0
        rows.append({'clase': c, 'soporte': tp + fn, 'predichas': tp + fp, 'precision': round(p, 3), 'recall': round(r, 3),
                     'f1': round(2 * p * r / (p + r), 3) if p + r else 0})
    rep = pd.DataFrame(rows)
    print(f'n = {len(y_true)} | exactitud = {acc:.3f} | macro-F1 = {rep.f1.mean():.3f}   (referencia con few-shot: 0.35)')
    display(rep)
    display(pd.crosstab(pd.Series(y_true, name='humano'), pd.Series(y_pred, name='modelo')))
    if probs is not None:
        d = pd.DataFrame({'ok': [a == b for a, b in zip(y_true, y_pred)], 'p': probs})
        d['tramo'] = pd.cut(d.p, [0, .5, .8, .95, 1.0001], labels=['<0.5', '0.5-0.8', '0.8-0.95', '>0.95'])
        display(d.groupby('tramo', observed=True)['ok'].agg(['mean', 'count']).rename(columns={'mean': 'exactitud'}))
    return rep

## 3. Carga del modelo, tokenización y entrenamiento

In [ ]:
import torch
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

assert torch.cuda.is_available(), 'No hay GPU CUDA visible'
print(torch.cuda.get_device_name(0), f'| VRAM libre: {torch.cuda.mem_get_info()[0] / 2**30:.1f} GiB')

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.padding_side = 'left'
if tok.pad_token is None: tok.pad_token = tok.eos_token

LETTER_IDS = [tok.encode(l, add_special_tokens=False) for l in LETTERS]
assert all(len(i) == 1 for i in LETTER_IDS), LETTER_IDS
LETTER_IDS = [i[0] for i in LETTER_IDS]
IM_END = tok.convert_tokens_to_ids('<|im_end|>')

def prompt_text(row):
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': user_message(row)}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def encode_example(row):
    p = tok(prompt_text(row), add_special_tokens=False)['input_ids']
    ans = [LETTER_IDS[LABELS.index(row['label'])], IM_END]
    return {'input_ids': p + ans, 'labels': [-100] * len(p) + ans}

class DS(torch.utils.data.Dataset):
    def __init__(self, df):
        ex = [encode_example(r) for _, r in df.iterrows()]
        self.items = [e for e in ex if len(e['input_ids']) <= MAX_LEN]
        print(f'{len(self.items)} ejemplos ({len(ex) - len(self.items)} descartados por > {MAX_LEN} tokens)')
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def collate(batch):
    n = max(len(b['input_ids']) for b in batch)
    ids  = torch.tensor([[tok.pad_token_id] * (n - len(b['input_ids'])) + b['input_ids'] for b in batch])
    lab  = torch.tensor([[-100] * (n - len(b['labels'])) + b['labels'] for b in batch])
    mask = torch.tensor([[0] * (n - len(b['input_ids'])) + [1] * len(b['input_ids']) for b in batch])
    return {'input_ids': ids, 'labels': lab, 'attention_mask': mask}

_ex = encode_example(train_os.iloc[0])
print('tokens del primer ejemplo:', len(_ex['input_ids']), '| respuesta:', tok.decode(_ex['input_ids'][-2:]), '=', train_os.iloc[0]['label'])

In [ ]:
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map={'': 0}, torch_dtype=torch.bfloat16)
print(f'modelo base cargado | VRAM libre: {torch.cuda.mem_get_info()[0] / 2**30:.1f} GiB')

### Entrenamiento

Si `qlora_citas_adapter_v1/` ya existe se carga en lugar de reentrenar (pon `FORCE_RETRAIN = True` para repetir). Si el entrenamiento se interrumpe, se reanuda desde el último checkpoint (checkpoint cada `SAVE_STEPS` pasos) con `RESUME = True`. El adaptador queda en `qlora_citas_adapter_v1/` y los checkpoints en `qlora_citas_trainer_v1/`.

In [ ]:
FORCE_RETRAIN = False
RESUME = True

if ADAPTER_DIR.exists() and not FORCE_RETRAIN:
    model = PeftModel.from_pretrained(model, ADAPTER_DIR)
    print('Adaptador cargado desde', ADAPTER_DIR)
else:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                            gradient_checkpointing_kwargs={'use_reentrant': False})
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']))
    model.print_trainable_parameters()

    train_ds = DS(train_os)
    total_steps = -(-len(train_ds) // (BATCH * GRAD_ACCUM)) * EPOCHS
    args = TrainingArguments(
        output_dir=str(TRAIN_DIR), per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type='cosine', warmup_steps=max(1, int(0.05 * total_steps)),
        logging_steps=10, bf16=True, optim='paged_adamw_8bit', gradient_checkpointing=True,
        save_strategy='steps', save_steps=SAVE_STEPS, save_total_limit=2, report_to='none', remove_unused_columns=False, seed=SEED)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collate)
    has_ckpt = TRAIN_DIR.exists() and any(TRAIN_DIR.glob('checkpoint-*'))
    trainer.train(resume_from_checkpoint=True if (RESUME and has_ckpt) else None)
    model.save_pretrained(ADAPTER_DIR)
    print('Adaptador guardado en', ADAPTER_DIR)

## 4. Inferencia: un solo *forward pass* y probabilidades sobre las 11 letras

In [ ]:
model.eval()

@torch.no_grad()
def predict_rows(rows):
    enc = tok([prompt_text(r) for r in rows], return_tensors='pt', padding=True, add_special_tokens=False).to('cuda')
    try:
        logits = model(**enc, use_cache=False, logits_to_keep=1).logits[:, -1, :]   # solo el último token: evita logits de todo el vocabulario
    except TypeError:
        logits = model(**enc, use_cache=False).logits[:, -1, :]                       # transformers antiguo: baja INFER_BATCH si hay OOM
    probs = logits[:, LETTER_IDS].float().softmax(-1).cpu()
    out = []
    for p in probs:
        k = int(p.argmax()); out.append({'label': LABELS[k], 'p': float(p[k]), 'dist': [round(float(x), 4) for x in p]})
    return out

def to_record(cid, pr):
    lab, p = pr['label'], pr['p']
    return {'context_id': cid, 'label': lab, 'p': round(p, 4), 'dist': pr['dist'],
            'evaluable': 'Evaluable' if lab in FUNCTIONS else lab,
            'funcion': lab if lab in FUNCTIONS else '',
            'confianza': 'Alta' if p >= CONF_HIGH else 'Media' if p >= CONF_MID else 'Baja'}

def load_done(path):
    done = {}
    if path.exists():
        for line in path.read_text(encoding='utf-8').splitlines():
            if line.strip():
                d = json.loads(line); done[d['context_id']] = d
    return done

def run_batch(df, path, max_rows=None):
    done = load_done(path)
    todo = df[~df['context_id'].isin(done)].copy()
    todo['_len'] = todo[COLS['ctx']].str.len() + todo[COLS['abs']].str.len().clip(upper=ABSTRACT_CHARS)
    todo = todo.sort_values('_len')                      # lotes de longitud parecida = menos padding
    if max_rows: todo = todo.head(max_rows)
    rows = [r for _, r in todo.iterrows()]
    print(f'{len(done)} ya hechos, {len(rows)} por procesar')
    t0 = time.time()
    with path.open('a', encoding='utf-8') as f:
        for i in range(0, len(rows), INFER_BATCH):
            chunk = rows[i:i + INFER_BATCH]
            for r, pr in zip(chunk, predict_rows(chunk)):
                f.write(json.dumps(to_record(r['context_id'], pr), ensure_ascii=False) + '\n')
            f.flush()
            n = i + len(chunk)
            if (i // INFER_BATCH) % 25 == 0:
                rate = n / (time.time() - t0)
                print(f'{n}/{len(rows)}  {rate:.1f} filas/s  ETA {(len(rows) - n) / rate / 60:.0f} min')
    return load_done(path)

## 5. Validación interna (solo para elegir hiperparámetros)

Esta validación está agrupada por citante, pero **no es el resultado a reportar**: el resultado final es el del Test Gold humano. Aquí compara con el baseline de few-shot (macro-F1 0.35, exactitud de función 0.44) y fíjate en las clases raras, en la diferencia entre citado visto y no visto, y en si la **probabilidad** separa los aciertos de los errores (tabla por tramos).

In [ ]:
assert not FINAL_FIT_ALL, 'Con FINAL_FIT_ALL=True el modelo vio la validación: estas métricas no son válidas.'
vp = run_batch(val_df, VAL_JSONL)
y_true = [r['label'] for _, r in val_df.iterrows()]
y_pred = [vp[c]['label'] for c in val_df['context_id']]
probs  = [vp[c]['p'] for c in val_df['context_id']]
print('=== TODA LA VALIDACIÓN ===')
_ = report(y_true, y_pred, probs)
for nombre, mask in [('CITADO NO VISTO EN TRAIN', ~val_df['cited_visto'].values), ('CITADO VISTO EN TRAIN', val_df['cited_visto'].values)]:
    idx = np.where(mask)[0]
    print(f'=== {nombre} ({len(idx)} casos) ===')
    _ = report([y_true[i] for i in idx], [y_pred[i] for i in idx])

**Cómo decidir:** si el macro-F1 en citado no visto supera claramente al baseline (≥ 0.5), pasa al paso 6. Si no: más épocas (`EPOCHS = 3`), `LORA_R = 32`, o revisar la calidad/consistencia de las anotaciones (los errores más frecuentes en la matriz de confusión suelen señalar categorías ambiguas en la guía). Con 2 700 casos y clases raras de ~50 ejemplos, la validación es ruidosa: no sobreajustes hiperparámetros a ella. Una vez fijados, puedes poner `FINAL_FIT_ALL = True`, borrar el estado del kernel y reentrenar con todo. Para recalibrar `Alta/Media/Baja`, ajusta `CONF_HIGH`/`CONF_MID` según la tabla por tramos (y re-ejecuta la exportación; las probabilidades guardadas no cambian).

## 6. Clasificar las 18 000 citas

Primero una tanda corta para medir velocidad; luego la corrida completa (reanudable: si se interrumpe, vuelve a ejecutar la celda).

In [ ]:
_ = run_batch(target, RUN_JSONL, max_rows=64)

In [ ]:
results = run_batch(target, RUN_JSONL)
print(len(results), 'de', len(target))

## 7. Exportar

In [ ]:
results = load_done(RUN_JSONL)
# los umbrales Alta/Media/Baja se recalculan desde la probabilidad guardada
def conf(p): return 'Alta' if p >= CONF_HIGH else 'Media' if p >= CONF_MID else 'Baja'

out = target.copy()
m = out['context_id'].map(results)
get = lambda k: m.map(lambda d: d[k] if isinstance(d, dict) else '')
out[COLS['ev']]   = get('evaluable')
out[COLS['fn']]   = get('funcion')
out[COLS['conf']] = m.map(lambda d: conf(d['p']) if isinstance(d, dict) else '')
out[COLS['just']] = m.map(lambda d: f"QLoRA p={d['p']:.2f}" if isinstance(d, dict) else '')
out['Estado']     = m.map(lambda d: ('COMPLETO' if d['evaluable'] == 'Evaluable' else 'REGISTRADO') if isinstance(d, dict) else '')
out.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')

print('filas sin clasificar:', (out[COLS['ev']] == '').sum())
print(out[COLS['ev']].value_counts().to_dict())
print(out[COLS['fn']].value_counts().to_dict())
print(out[COLS['conf']].value_counts().to_dict())
print('Filas a revisar a mano (Confianza Baja o no Evaluable):', ((out[COLS['conf']] == 'Baja') | (out[COLS['ev']] != 'Evaluable')).sum())